In [1]:
import sys
from pathlib import Path

# Path to the root directory of the repository
root_dir = Path().resolve().parent
sys.path.append(str(root_dir / 'src' / 'python'))

import numpy as np
import matplotlib.pyplot as plt
from pybamm import Experiment
import dandeliion.client as dandeliion

# The following lines enable logging output in this notebook
import logging
logging.basicConfig(stream=sys.stdout, level=logging.INFO)

In [2]:
print(dandeliion.__file__)

/Users/maximedeckers/Documents/RSE/dandeliion/dandeliion-client/src/python/dandeliion/client/__init__.py


### Authentication
One of the ways to define the simulator object can be the following:

In [3]:
api_url = "http://localhost/v1"
#api_key = "some_hash"

#api_url = "https://api.dandeliion.com/v1"
api_key = "f1543b9ff2001d04bd2c002d9375c8a28ad0bcac9914ce5d8f0e0b98ef4404cf"

simulator = dandeliion.Simulator(api_url, api_key)

# The following sets up a mock simulator (to be used instead of actual simulator for testing)
sys.path.append(str(root_dir / 'tests' / 'python' / 'dandeliion' / 'client'))

# from mock_simulator import MockSimulator
# simulator = MockSimulator(api_url, api_key)

In [4]:
simulator

Simulator(api_url='http://localhost/v1', api_key='f1543b9ff2001d04bd2c002d9375c8a28ad0bcac9914ce5d8f0e0b98ef4404cf')

### Parameters
A valid BPX file with battery cell parameters (json-file)

In [5]:
# Can be a file or already read-in valid BPX as dict or BPX object
params = root_dir / 'examples' / 'AE_gen1_BPX.json'

### Experiment
PyBaMM Experiment object, for example:

In [6]:
experiment = Experiment(
    [
        (
            "Discharge at 6 A for 100 seconds",
            "Rest for 10 seconds",
            "Charge at 1 A for 10 seconds",
        ),
        (
            "Discharge at 1 A for 20 seconds",
            "Rest for 10 seconds",
            "Charge at 1 A for 20 seconds",
        ),
    ] * 2,
    period="1 second",  # Optional
)

### Additional model/backend-specific parameters
A dictionary

In [7]:
extra_params = {}

# Number of mesh points in PyBaMM format (default is 16 mesh points everywhere):
extra_params['Mesh'] = {"x_n": 16, "x_s": 8, "x_p": 16, "r_n": 16, "r_p": 16}

# Initial state of charge (default is 1):
extra_params['Initial SOC'] = 1.0

In [8]:
params

PosixPath('/Users/maximedeckers/Documents/RSE/dandeliion/dandeliion-client/examples/AE_gen1_BPX.json')

### Calculating solution

In [9]:
solution = dandeliion.solve(
    simulator=simulator,
    params=params,
    experiment=experiment,  # Optional, default is 1C discharge. Translates into the "DandeLiion: Experiment" section
    extra_params=extra_params,
    is_blocking=True,
)
print(solution.status)

/Users/maximedeckers/opt/anaconda3/envs/dev-env/lib/python3.9/site-packages/pydantic/_internal/_decorators_v1.py:171: UserWarning: The minimum voltage computed from the STO limits (2.4988861558265327 V) is less than the lower voltage cut-off (2.5 V) with the absolute tolerance v_tol = 0.001 V
  model_dict_new = validator(model_dict)


INFO:dandeliion.client.simulator:
INFO:dandeliion.client.simulator:[2025-04-10 10:13:21.800] [async_file_logger] [info] ================================================================================
[2025-04-10 10:13:21.800] [async_file_logger] [info] Initialised asynchronous file logger...
[2025-04-10 10:13:21.810] [async_file_logger] [info] Parsed parameters:
[2025-04-10 10:13:21.810] [async_file_logger] [info] - Header:
[2025-04-10 10:13:21.810] [async_file_logger] [info] -- BPX version: 0.3
[2025-04-10 10:13:21.810] [async_file_logger] [info] -- Title: About:Energy Gen1 Demo 21700 NMC811|graphite 5 Ah cell
[2025-04-10 10:13:21.810] [async_file_logger] [info] -- Description: About:Energy Gen1 Demo 21700 NMC811|graphite 5 Ah cell physics-based parameterisation. This is a fictitious cell used for technology demonstration. Some data are taken from literature: Nyman et al. 2008 (doi:10.1016/j.electacta.2008.04.023); Chen et al. 2020 (doi:10.1149/1945-7111/ab9050); O'Regan et al. 2022 

In [10]:
print(solution.status)

success


In [11]:
solution.log_file


KeyError: 'log_file'

In [ ]:
import json

# Convert solution to a dictionary using indexing
solution_dict = {key: solution[key] for key in solution.keys()}

# Write to file
with open("solution.json", "w") as f:
    json.dump(solution_dict, f, indent=4)

In [23]:
solution

### Results
Print all available keys in the solution object.
Here method `keys()` is used to get all available keys in the solution object.

In [24]:
for key in sorted(solution.keys()):
    print(key)

In [ ]:
# Print the final values of time, voltage, and temperature
print(f"Final time [s]: {solution['Time [s]'][-1]}")
print(f"Final voltage [V]: {solution['Voltage [V]'][-1]}")
print(f"Final temperature [K]: {solution['Temperature [K]'][-1]}")

In [ ]:
# Plot current and voltage vs time.
# Here we access scalar values vs time.
fig, axs = plt.subplots(2, 1, figsize=(10, 8))
axs[0].plot(solution["Time [s]"], solution["Current [A]"], label="Dandeliion")
axs[0].set_xlabel("time [s]")
axs[0].set_title("Current [A]")
axs[0].legend()
axs[0].grid()
axs[1].plot(solution["Time [s]"], solution["Voltage [V]"], label="Dandeliion")
axs[1].set_xlabel("time [s]")
axs[1].set_title("Voltage [V]")
axs[1].legend()
axs[1].grid()
plt.tight_layout()
plt.show()

In [ ]:
# Concentration in the electrolyte vs `x` at the last time step.
# Here we access spatially dependent values vs time.
plt.plot(
    solution["Electrolyte x-coordinate [m]"] * 1e6,
    solution["Electrolyte concentration [mol.m-3]"][-1],
    label="Dandeliion",
)
plt.xlabel(r"x [$\mu$m]")
plt.title("Electrolyte conc. (end of experiment) [mol.m-3]")
plt.legend()
plt.grid()
plt.show()

In [ ]:
solution["Electrolyte x-coordinate [m]"][-1]

In [ ]:
# A list of output times (optional)
t_eval = np.arange(0, 3600, 1)

# If the user needs the solution at the `t_eval` times, the following code can be used (works only correctly on columns with timeline data)
# This is a linear interpolation with constant extrapolation
solution["Voltage [V]"](t=t_eval)